In [ ]:
!git clone https://github.com/cfzd/Ultra-Fast-Lane-Detection.git
%cd Ultra-Fast-Lane-Detection

In [ ]:
!pip install -q opencv-python tqdm tensorboard addict scikit-learn pathspec

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q gdown

!rm models/tusimple_18.pth

!gdown 1WCYyur5ZaWczH15ecmeDowrW30xcLrCn -O models/tusimple_18.pth

In [ ]:
from model.model import parsingNet
from utils.common import merge_config

print("Official UFLD repo imports working!")

## Load pretrained UFLD model and prepare for inference

In [ ]:
import torch
from model.model import parsingNet

device = "cuda"

net = parsingNet(
    pretrained=False,
    backbone="18",
    cls_dim=(101, 56, 4),
    use_aux=False
)

checkpoint = torch.load(
    "models/tusimple_18.pth",
    map_location="cpu"
)

state_dict = checkpoint["model"]

# remove module. prefix if present
new_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("module."):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

net.load_state_dict(new_state_dict, strict=False)

net.to(device)
net.eval()

print("Official UFLD model loaded!")

## Load dashcam video and extract a sample frame for testing

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/dashcam_video.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

print("Frame loaded:", ret)
print("Shape:", frame.shape)

cap.release()

plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")

## Preprocess input frame for UFLD model inference

In [ ]:
import torchvision.transforms as transforms
from PIL import Image
import torch
import numpy as np
import cv2

img_transforms = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

# convert frame BGR -> RGB -> PIL
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

pil_img = Image.fromarray(frame_rgb)

input_tensor = img_transforms(pil_img)

# add batch dimension
input_tensor = input_tensor.unsqueeze(0).cuda()

print(input_tensor.shape)

## Run UFLD model inference and obtain lane prediction output

In [ ]:
with torch.no_grad():
    output = net(input_tensor)


## Decode UFLD model output into lane coordinates

In [ ]:
import scipy.special
import numpy as np

# Tusimple settings
griding_num = 100
img_w = 3840
img_h = 2160

# import anchor points
from data.constant import tusimple_row_anchor

cls_num_per_lane = 56

out_j = output[0].data.cpu().numpy()

# same as official demo
out_j = out_j[:, ::-1, :]

prob = scipy.special.softmax(
    out_j[:-1, :, :],
    axis=0
)

idx = np.arange(griding_num) + 1
idx = idx.reshape(-1, 1, 1)

loc = np.sum(prob * idx, axis=0)

out_j = np.argmax(out_j, axis=0)

loc[out_j == griding_num] = 0

lane_points = []

col_sample = np.linspace(0, 800 - 1, griding_num)
col_sample_w = col_sample[1] - col_sample[0]


for lane_num in range(loc.shape[1]):

    points = []

    if np.sum(loc[:, lane_num] != 0) > 2:

        for point_num in range(loc.shape[0]):

            if loc[point_num, lane_num] > 0:

                x = int(
                    loc[point_num, lane_num] *
                    col_sample_w *
                    img_w / 800
                ) - 1

                y = int(
                    img_h *
                    (
                        tusimple_row_anchor[
                            cls_num_per_lane - 1 - point_num
                        ] / 288
                    )
                ) - 1

                points.append((x,y))

    lane_points.append(points)


print("Detected lanes:", len(lane_points))

## Visualize detected lane points and remove bonnet region

In [ ]:
import cv2
import matplotlib.pyplot as plt

output_frame = frame.copy()

HOOD_Y_START = 1400   # we can tune this

for lane in lane_points:
    filtered_points = []

    for x, y in lane:
        if y < HOOD_Y_START:
            filtered_points.append((x,y))

    for x,y in filtered_points:
        cv2.circle(output_frame, (x,y), 10, (0,255,0), -1)


plt.figure(figsize=(16,9))
plt.imshow(cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")

## Apply UFLD lane detection on complete dashcam video

In [ ]:
import cv2
import torch
import scipy.special
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm


# Video paths
video_path = "/content/dashcam_video.mp4"
output_path = "/content/ufld_output.mp4"


# Remove bonnet area
HOOD_Y_START = 1400

# Transform
img_transforms = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

# Video setup
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)


# Tusimple settings
griding_num = 100
cls_num_per_lane = 56
img_w = width
img_h = height

from data.constant import tusimple_row_anchor

# Process video
for _ in tqdm(range(frames)):

    ret, frame = cap.read()

    if not ret:
        break


    output_frame = frame.copy()


    # preprocessing
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)

    input_tensor = img_transforms(pil_img)
    input_tensor = input_tensor.unsqueeze(0).cuda()


    # inference
    with torch.no_grad():
        output = net(input_tensor)

    # Decode output
    out_j = output[0].cpu().numpy()

    out_j = out_j[:, ::-1, :]


    prob = scipy.special.softmax(
        out_j[:-1,:,:],
        axis=0
    )


    idx = np.arange(griding_num) + 1
    idx = idx.reshape(-1,1,1)


    loc = np.sum(prob * idx, axis=0)


    out_j = np.argmax(out_j, axis=0)

    loc[out_j == griding_num] = 0


    col_sample = np.linspace(
        0,
        799,
        griding_num
    )

    col_sample_w = col_sample[1] - col_sample[0]

    # Draw lanes
    for lane_num in range(loc.shape[1]):

        if np.sum(loc[:, lane_num] != 0) > 2:

            for point_num in range(loc.shape[0]):

                if loc[point_num, lane_num] > 0:

                    x = int(
                        loc[point_num, lane_num]
                        * col_sample_w
                        * img_w / 800
                    ) - 1


                    y = int(
                        img_h *
                        (
                        tusimple_row_anchor[
                            cls_num_per_lane-1-point_num
                        ] / 288
                        )
                    ) - 1


                    # remove bonnet points
                    if y < HOOD_Y_START:
                        cv2.circle(
                            output_frame,
                            (x,y),
                            5,
                            (0,255,0),
                            -1
                        )


    writer.write(output_frame)


cap.release()
writer.release()


print("Done!")
print("Saved:", output_path)

In [ ]:
from google.colab import files

files.download("/content/ufld_output.mp4")